# IPC2BNS-Verify — Phase 3: Generation Layer & Stage 1 / Stage 2 Ablation

This notebook demonstrates and evaluates the generative answering layer:
1. **Legal Prompt Builder** (`prompt_template.py`): Enforces canonical statutory citations `[Act §Section]`.
2. **Stage 1 (Baseline LLM, Closed-Book)**: Evaluates baseline generative model without retrieval augmentation.
3. **Stage 2 (+RAG Context)**: Evaluates generative model augmented with top-k retrieved bare-act chunks.
4. **Citation Extraction & Comparison**: Compares hallucinations in Stage 1 vs. statutory grounding in Stage 2.
5. **Automated Unit Tests**: Runs the 55-test pytest suite.

---
## 1. Mount Google Drive & Environment Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json
PROJECT_ROOT = '/content/drive/MyDrive/NLP_rspaper'
os.environ['IPC2BNS_PROJECT_ROOT'] = PROJECT_ROOT

if os.path.join(PROJECT_ROOT, 'code') not in sys.path:
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'code'))

print('Project Root:', PROJECT_ROOT)
print('Environment configured.')

Mounted at /content/drive
Project Root: /content/drive/MyDrive/NLP_rspaper
Environment configured.


---
## 2. Install Pytest

In [2]:
!pip install -q pytest
print('Pytest ready.')

Pytest ready.


---
## 3. Prompt Construction & Citation Format Demo

In [3]:
from src.generation.prompt_template import LegalPromptBuilder
from src.retrieval.search import retrieve_statutes

query = 'What is the punishment for cheating under BNS 2023?'
chunks = retrieve_statutes(query, top_k=2, act_filter='BNS')

stage2_prompt = LegalPromptBuilder.build_stage2_prompt(query, chunks)
print('=== CONSTRUCTED STAGE 2 SYSTEM PROMPT ===')
print(stage2_prompt['system_prompt'])
print('\n=== CONSTRUCTED STATUTORY CONTEXT & USER PROMPT ===')
print(stage2_prompt['user_prompt'][:500] + '...')

=== CONSTRUCTED STAGE 2 SYSTEM PROMPT ===
You are an authoritative Indian legal assistant specializing in the transition from the Indian Penal Code (IPC 1860) to the Bharatiya Nyaya Sanhita (BNS 2023).

Instructions:
1. You are provided with AUTHORITATIVE STATUTORY CONTEXT below retrieved from the official bare acts.
2. Answer the user's question relying STRICTLY and EXCLUSIVELY on the provided statutory context.
3. For EVERY substantive assertion, cite the exact provision from the context using the format: [Act §SectionNumber], for example: [BNS §103] or [IPC §302].
4. Do NOT cite any section number that is not explicitly present in the provided context.
5. If the provided context does not contain sufficient information or if a section was repealed, explicitly state: "Based on statutory context, this provision is repealed / not present."
6. Be concise, objective, and precise.

=== CONSTRUCTED STATUTORY CONTEXT & USER PROMPT ===
=== STATUTORY CONTEXT ===
--- STATUTORY PROVISION #1 ---


---
## 4. Compare Stage 1 (Baseline) vs Stage 2 (+RAG)

In [4]:
from src.generation.generator import get_generator

generator = get_generator()
test_queries = [
    'What is the punishment for murder under BNS?',
    'Where is dowry death covered in the new law?',
    'What happened to sedition under Section 124A IPC in BNS 2023?',
]

for q in test_queries:
    print('='*75)
    print(f'QUESTION: {q}')
    print('='*75)

    res1 = generator.generate_stage1(q)
    print(f'\n[STAGE 1 — Baseline LLM (No Context)]')
    print(f'Answer   : {res1.generated_text}')
    print(f'Citations: {[c["raw"] for c in res1.citations]}')

    res2 = generator.generate_stage2(q, top_k=2)
    print(f'\n[STAGE 2 — +RAG (Retrieved Statutory Context)]')
    print(f'Answer   : {res2.generated_text}')
    print(f'Citations: {[c["raw"] for c in res2.citations]}')
    print()

QUESTION: What is the punishment for murder under BNS?

[STAGE 1 — Baseline LLM (No Context)]
Answer   : Under Indian criminal law, murder is penalized under [IPC §302] with death or life imprisonment. Under the new Bharatiya Nyaya Sanhita, it has been renumbered to [BNS §103].
Citations: ['[IPC §302]', '[BNS §103]']

[STAGE 2 — +RAG (Retrieved Statutory Context)]
Answer   : Based on authoritative statutory context, this matter is governed by [IPC §302] (Punishment for murder). Statutory provision: Whoever commits murder shall be punished with death, or imprisonment for life, and shall also be liable to fine. Additionally, [BNS §103] (Punishment for murder) is relevant.
Citations: ['[IPC §302]', '[BNS §103]']

QUESTION: Where is dowry death covered in the new law?

[STAGE 1 — Baseline LLM (No Context)]
Answer   : Dowry death is punished with a minimum of seven years imprisonment under [IPC §304B] and now [BNS §80].
Citations: ['[IPC §304B]', '[BNS §80]']

[STAGE 2 — +RAG (Retrieved Sta

---
## 5. Execute Full Benchmark Ablations (Stage 1 & Stage 2)

In [5]:
from src.generation.run_ablations import run_stage1_ablation, run_stage2_ablation

benchmark_dev = os.path.join(PROJECT_ROOT, 'data/03_benchmark/benchmark_dev.csv')
stage1_out = os.path.join(PROJECT_ROOT, 'results/stage1/stage1_baseline_results.json')
stage2_out = os.path.join(PROJECT_ROOT, 'results/stage2/stage2_rag_results.json')

run_stage1_ablation(benchmark_dev, stage1_out)
run_stage2_ablation(benchmark_dev, stage2_out)

print('\nBoth Stage 1 and Stage 2 ablation results generated successfully.')


Both Stage 1 and Stage 2 ablation results generated successfully.


---
## 6. Inspect Results Summary

In [6]:
with open(stage1_out, 'r') as f:
    s1_data = json.load(f)
with open(stage2_out, 'r') as f:
    s2_data = json.load(f)

print(f'Stage 1 queries completed: {len(s1_data["results"])}')
print(f'Stage 2 queries completed: {len(s2_data["results"])}')

# Sample output comparison
s1_sample = s1_data['results'][0]
s2_sample = s2_data['results'][0]
print('\n--- Sample Query Comparison ---')
print(f'Query        : {s1_sample["query_text"]}')
print(f'Ground Truth : {s1_sample["ground_truth_sections"]} - {s1_sample["ground_truth_answer"][:80]}...')
print(f'Stage 1 Cited: {s1_sample["cited_sections"]}')
print(f'Stage 2 Cited: {s2_sample["cited_sections"]}')

Stage 1 queries completed: 17
Stage 2 queries completed: 17

--- Sample Query Comparison ---
Query        : What is the new section for murder under the Bharatiya Nyaya Sanhita, 2023?
Ground Truth : 103 - Under BNS 2023, punishment for murder is governed by Section 103 (previously Sec...
Stage 1 Cited: ['302', '103']
Stage 2 Cited: ['101', '103']


---
## 7. Run Full Automated Test Suite (55 Tests)

In [7]:
test_dir = os.path.join(PROJECT_ROOT, 'code/tests')
!python -m pytest "{test_dir}" -v --color=yes

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: typeguard-4.6.0, langsmith-0.11.1, anyio-4.14.2
collected 55 items                                                             

drive/MyDrive/NLP_rspaper/code/tests/test_concordance.py::test_concordance_table_loads_successfully PASSED [  1%]
drive/MyDrive/NLP_rspaper/code/tests/test_concordance.py::test_concordance_schema_columns PASSED [  3%]
drive/MyDrive/NLP_rspaper/code/tests/test_concordance.py::test_deterministic_exact_mappings[302-103] PASSED [  5%]
drive/MyDrive/NLP_rspaper/code/tests/test_concordance.py::test_deterministic_exact_mappings[299-100] PASSED [  7%]
drive/MyDrive/NLP_rspaper/code/tests/test_concordance.py::test_deterministic_exact_mappings[304A-106] PASSED [  9%]
drive/MyDrive/NLP_rspaper/code/tests/test_concordance.py::test_deterministic_exact_mappings[30

---
## 8. Check Progress against WBS

In [8]:
!python "{PROJECT_ROOT}/check_progress.py" --root "{PROJECT_ROOT}" --write-report

# Project Progress Report
**Overall: 17/32 tasks complete (53%)**

_Generated: 2026-09-03T07:14:44_

## 0. Setup — 3/4 (75%)
- [x] Repo scaffolding + config system  `(code/src, code/configs)`
- [x] India Code raw text downloaded  `(data/00_raw/india_code)`
- [ ] Concordance source PDF(s) collected  `(data/00_raw/concordance_source_pdfs)`
- [x] Data Management Plan written  `(docs/IPC2BNS-Verify_Data_Management_Plan.md)`

## 1. Mapping Module — 5/5 (100%)
- [x] Ground-truth concordance table finalized  `(data/02_ground_truth/concordance_v1.csv)`
- [x] Concordance validation report reviewed  `(data/02_ground_truth/validation_report.csv)`
- [x] Deterministic lookup function implemented  `(code/src/mapping/lookup.py)`
- [x] Query normalizer implemented  `(code/src/mapping/normalizer.py)`
- [x] Mapping module unit tests  `(code/tests/test_concordance.py)`

## 2. Ingestion & Retrieval — 6/6 (100%)
- [x] Section-level chunker implemented  `(code/src/ingestion/chunker.py)`
- [x] Cleaned sectio